In [ ]:
%pip install -U peft bitsandbytes transformers accelerate

In [ ]:
%pip install PyMuPDF

In [ ]:
%pip install -U trl

In [16]:
from datasets import Dataset, load_dataset

In [ ]:
# Non Instruction Fine tuning datasets 
dataset = load_dataset("HuggingFaceFW/fineweb")
pubmed = load_dataset("ncbi/pubmed")
dataset = load_dataset("datajuicer/the-pile-pubmed-abstracts-refined-by-data-juicer")
dataset = load_dataset("open-llm-leaderboard/open_llm_corpus")
owt = load_dataset("Skylion007/openwebtext")
ds = load_dataset("armanc/scientific_papers")

### Custom Dataset

In [5]:
from google.colab import files
upload = files.upload()

Saving Metformin.pdf to Metformin.pdf


In [6]:
import fitz
def ExtractText(pdf_path):
  textData = []
  with fitz.open(pdf_path) as doc:
    for page in doc:
      text = page.get_text("text").strip()
      if text:
        textData.append(text)
  return textData

In [7]:
pdfPath = "/content/Metformin.pdf"
content = ExtractText(pdfPath)
content

['Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis. \n \nClinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA red

In [8]:
# Splitting
import re
def ChunkDataset(pages):
  chunksData = []
  for page in pages:
    chunks = re.split(r'\n\s*\n', page)
    for chunk in chunks:
      cleanedText = chunk.strip()
      if len(cleanedText) > 30:
        chunksData.append(cleanedText)
  return chunksData

In [9]:
chunks = ChunkDataset(content)
print(len(chunks))
print(chunks[0])

4
Metformin is one of the most widely prescribed oral antihyperglycemic agents.​
 Its primary mechanism of action involves the activation of AMP-activated protein kinase 
(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation 
while inhibiting hepatic gluconeogenesis.​
 Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes 
and display anti-inflammatory properties.​
 Recent studies also suggest potential anticancer effects through inhibition of the mTOR 
signaling pathway and suppression of tumor angiogenesis.


In [10]:
data = [{"text" : chunk} for chunk in chunks]
for d in data:
  print(d)
  print()

{'text': 'Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis.'}

{'text': 'Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits hep

In [11]:
# Model Selection
model_name = "Qwen/Qwen2.5-0.5B"

In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling

In [13]:
tokenizer = AutoTokenizer.from_pretrained(model_name) # load the correct tokenizer for the model
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [14]:
def TokenizeText(data):
  tokens = tokenizer(data["text"], padding="max_length", truncation=True, max_length=512)
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens

In [17]:
customDataset = Dataset.from_list(data)
customDataset

Dataset({
    features: ['text'],
    num_rows: 4
})

In [19]:
tokenizedData = customDataset.map(TokenizeText, batched = True, remove_columns = ["text"])

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [20]:
tokenizedData

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4
})

In [21]:
model = AutoModelForCausalLM.from_pretrained(model_name)

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [22]:
from transformers import TrainingArguments
help(TrainingArguments)

Help on class TrainingArguments in module transformers.training_args:

class TrainingArguments(builtins.object)
 |  TrainingArguments(output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 5e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool = False, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing

In [23]:
trainingArgs = TrainingArguments(
    output_dir = "./Qwen-pharma-domain",
    num_train_epochs = 1,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=2e-5,
    fp16=True,
    report_to="none"
)

In [ ]:
# This will train whole model -> out of memory error
trainer = Trainer(
    model = model,
    args = trainingArgs,
    train_dataset = tokenizedData
)

trainer.train()

### Partial Finetuning Method
* Method 1: Feerze some layer and finetune unfreeze layer(old CNN and Bert sytel method)

* Method 2: LORA(Append some external weight to the already trained pretrain weight)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset
import os, re

model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

# Freeze all layers
for param in model.parameters():
     param.requires_grad = False

# Unfreeze last 4 transformer blocks + lm_head
for name, param in model.named_parameters():
    if any(f"model.layers.{i}." in name for i in range(20, 24)):
        param.requires_grad = True
    elif "lm_head" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable/total*100:.2f}% of total parameters")


dataset = load_dataset("json", data_files={"train": "pharma_non_instruction.jsonl"})

def TokenizeText(data):
  tokens = tokenizer(data["text"], padding="max_length", truncation=True, max_length=512)
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens

tokenized = customDataset["train"].map(TokenizeText, batched=True, remove_columns=["text"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

os.environ["WANDB_DISABLED"] = "true"
training_args = TrainingArguments(
     output_dir="./tinyllama-pharma-last4",
     overwrite_output_dir=True,
     num_train_epochs=2,
     per_device_train_batch_size=2,
     gradient_accumulation_steps=4,
     learning_rate=1e-4,
     fp16=True,
     logging_steps=20,
     save_steps=200,
     save_total_limit=2,
     report_to="none"
)


trainer = Trainer(
     model=model,
     args=training_args,
     train_dataset=tokenized,
     data_collator=data_collator
)

trainer.train()
trainer.save_model("./Qwen-pharma-last4-final")
print("\nTraining completed and model saved!")


* Method 2

In [25]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
!pip install transformers accelerate bitsandbytes peft datasets

In [33]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [34]:
model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [37]:
def TokenizeText(data):
  tokens = tokenizer(data["text"], padding="max_length", truncation=True, max_length=512)
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens

tokenized = customDataset.map(TokenizeText, batched=True, remove_columns=["text"])

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [44]:
from transformers import BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [45]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [49]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none"
)
non_inst_model_lora = get_peft_model(model, lora_config)

In [59]:
args = TrainingArguments(
    output_dir="./Qwen-lora",
    num_train_epochs=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

In [60]:
trainer = Trainer(
    model=non_inst_model_lora,
    args=args,
    train_dataset=tokenized
)

trainer.train()

Step,Training Loss


TrainOutput(global_step=10, training_loss=0.4985204696655273, metrics={'train_runtime': 9.045, 'train_samples_per_second': 4.422, 'train_steps_per_second': 1.106, 'total_flos': 44044957777920.0, 'train_loss': 0.4985204696655273, 'epoch': 10.0})

In [62]:
import os
print(os.listdir("/content"))
print(os.listdir("/content/Qwen-lora"))

['.config', 'Qwen-lora', 'Qwne-lora', 'Metformin.pdf', 'tinyllama-lora', 'sample_data']
['checkpoint-10']


In [63]:
trainedModelPath = "/content/Qwen-lora"
from peft import PeftModel
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

trainedModel = PeftModel.from_pretrained(
    base_model,
    "/content/Qwen-lora/checkpoint-10"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
trainedModel.print_trainable_parameters()

In [65]:
prompt = "Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



Model Output:

Clinical trials demonstrated that combining Atorvastatin with Ezetimibe provided a significant reduction in the risk of cardiovascular events, including strokes and heart attacks. Which of the following mechanisms is most likely responsible for this effect?
A. Reduced absorption of cholesterol by the intestine
B. Increased clearance of low-density lipoprotein (LDL)
C. Enhanced absorption of bile acids in the small intestine
D. Elevated plasma concentrations of apolipoproteins involved in lipid metabolism
E. Decreased synthesis of endogenous adipokines
Answer:

D

(Multiple
